### Data parsing

In [32]:
from langchain_core.documents import Document 

In [33]:
doc=Document(
    page_content="you gotta check yourself before you wreck yourself",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author":"damru",
        "date_created":"2026-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'damru', 'date_created': '2026-01-01'}, page_content='you gotta check yourself before you wreck yourself')

In [34]:
import os
os.makedirs("../data/files",exist_ok=True)

In [35]:
text={
    "../data/files/intro.txt":"""I have worked as a Software engineer for 2 years in India and 3 years in the US and during this time span I’ve realized that promotions, raises, and strong performance ratings are rarely about working the hardest. They’re about creating visible, meaningful IMPACT however small or big your organization is.

I have worked at a startUp in India and now I work at Walmart, and based on my experience I have compiled principles that have consistently worked for me. None of this is company-specific. These are patterns that apply almost everywhere.

1. Integrate AI Into Your Day-to-Day Work
Every company today is looking for employees who can use AI to save time, reduce manual effort, and increase team efficiency. These are also the people who tend to be most valuable and are safe during lay-offs.

As an employee, start by identifying where your team spends the most time:

Deployments

Production releases

Manual validations

Debugging

Documentation

Repetitive analysis

Then ask a simple question:
“What part of this can be automated or accelerated using AI?”

Start small. Even if the first solution only helps you, that’s fine. Over time, turn it into something that benefits the team. The moment your work starts saving other people time, you create leverage.

Eg: Initially, you can just create an agent.md file and integrate it with cursor or copilot, for it to work like a chatbot. You can integrate this chatbot to places where product can use it to make certain repetitive and simple changes, where engineering involvement is not necessary. This will help you save engineering bandwidth.

One honest truth: You will likely need to use your weekends or personal time early on to experiment and build these things. But believe me it will be WORTH IT.

2. Choose the Right Business Initiatives
During sprint planning or roadmap discussions, I stopped asking:
“What task should I pick up?”

And started asking: “Which initiative will make the biggest impact?”

Always evaluate work from an impact perspective, not an input perspective. Promotions and raises are not tied to how busy you were they’re tied to what changed because of your work.

If you’re early in your career, you may not be given ownership of major initiatives immediately. So initially you can:

Contribute beyond your assigned scope

Support high-impact projects

Just try to be involved in these projects in some way. The closer your work is to business priorities, the easier it becomes to justify your growth during reviews.

3. Become the Point of Contact (POC)
Growth in any company requires cross-functional visibility. This means interacting not just with other developers, but also with:

Product managers

Stakeholders

Sister teams and their leads

Data scientists

Support teams, etc.

When you become the POC for an application or system, people know:

What you’re responsible for

What problems you solve

Whom to reach out to

Year-end reviews often include feedback from these people, and if you are the POC for them, their feedback automatically will be in favor of your growth!

4. Communicate Your Impact Clearly
Early on in my career, I have made this mistake of assuming that my work will speak for itself. But it DOES NOT! Hence, in order to create visibility, I’ve made it a habit to track:

Problems I solved

Risks I prevented

Metrics I moved

Cross-team contributions

When I have 1-1 meetings with my managers or leads, I clearly state the impact: I focus less on what I did and more on what changed: Reduced failures, Improved performance, Saved time or cost, Increased adoption or reliability.

I also make it a point to write everything down well in advance of the meeting and discuss each point clearly. This helps keep the conversation focused and meaningful, rather than turning into a generic discussion without clear direction.

Your manager needs concrete examples to represent you well for promotions, and you explicitly repeating your impact, helps him/her remember it and use it to present your case.

"""
}

In [36]:
for f,c in text.items():
    with open(f,'w',encoding="utf-8") as f:
        f.write(c)
print("done")

done


In [37]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader
loader=DirectoryLoader("../data/files",
glob="**/*.txt",
loader_cls=TextLoader,
loader_kwargs={'encoding':'utf-8'},
show_progress=True
)
docs=loader.load()
print(docs)

100%|██████████| 1/1 [00:00<00:00, 42.85it/s]

[Document(metadata={'source': '..\\data\\files\\intro.txt'}, page_content='I have worked as a Software engineer for 2 years in India and 3 years in the US and during this time span I’ve realized that promotions, raises, and strong performance ratings are rarely about working the hardest. They’re about creating visible, meaningful IMPACT however small or big your organization is.\n\nI have worked at a startUp in India and now I work at Walmart, and based on my experience I have compiled principles that have consistently worked for me. None of this is company-specific. These are patterns that apply almost everywhere.\n\n1. Integrate AI Into Your Day-to-Day Work\nEvery company today is looking for employees who can use AI to save time, reduce manual effort, and increase team efficiency. These are also the people who tend to be most valuable and are safe during lay-offs.\n\nAs an employee, start by identifying where your team spends the most time:\n\nDeployments\n\nProduction releases\n\nM

In [38]:
from langchain_community.document_loaders import PyMuPDFLoader
loader=DirectoryLoader("../data/pdf",
glob="**/*.pdf",
loader_cls= PyMuPDFLoader,
show_progress=True
)
docs=loader.load()
print(docs)

100%|██████████| 1/1 [00:00<00:00,  5.52it/s]

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'file_path': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'format': 'PDF 1.4', 'title': 'Untitled document', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Importance of SAFERTOS (Safety-Critical \nReal-Time OS) \nSAFERTOS is a pre-certified, safety-critical RTOS designed for systems where determinism, \nreliability, and certification matter (automotive, medical devices, industrial safety).\u200b\n It is derived from FreeRTOS, but rewritten by safety experts and certified to IEC 61508 SIL3, \nISO 26262 ASIL D, IEC 62304 Class C, and more. \n1) SAFERTOS for Automotive and Supported Processors \nWhy SAFERTOS Is Used in Automotive Systems\u200b\nAutomotive systems must meet extreme safety and timing guarantees. SAFERTOS 

### Chunking

In [39]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

Found 1 PDF files to process

Processing: Untitled document (4) (1).pdf
  ✓ Loaded 5 pages

Total documents loaded: 5


[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

In [40]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs
chunks=split_documents(all_pdf_documents)
chunks

Split 5 documents into 6 chunks

Example chunk:
Content: Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
...
Metadata: {'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

### Embeddings and vector storage

In [41]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [42]:
class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully")
            print(f'Embedding dimension: {self.model.get_sentence_embedding_dimension()}')
        except Exception as e:
            raise Exception(f"Failed to load model: {e}")

    def generate_embeddings(self,texts: List[str]) ->np.ndarray:
        if not self.model:
            raise Exception("Model not loaded")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated {len(embeddings)} embeddings")
        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1525.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model sentence-transformers/all-MiniLM-L6-v2 loaded successfully
Embedding dimension: 384


### Vector search and similarity

In [43]:
class VectorStore:
    def __init__(self,collection_name: str="pdf_documents",persist_directory: str="./data/chroma_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,metadata={"description": "cosine similarity"}
            )
            print("Vector store initialized successfully")
            print(f"Collection: {self.collection_name}")
            print(f'collection count: {self.collection.count()}')
        except Exception as e:
            print(f"Error initializing store: {e}")
            raise
    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        print(f"adding {len(documents)} documents to vector store")
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list = []
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["source"]=doc.metadata.get("source","unknown")
            metadata["page"]=doc.metadata.get("page",0)
            metadata["chunk_index"]=i
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                documents=documents_text,
                metadatas=metadatas
            )
            print(f"Successfully added {len(documents)} documents to vector store")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise
vector_store = VectorStore()
vector_store 




Vector store initialized successfully
Collection: pdf_documents
collection count: 6


In [44]:
chunks

[Document(metadata={'producer': 'Skia/PDF m144 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '..\\data\\pdf\\Untitled document (4) (1).pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'Untitled document (4) (1).pdf', 'file_type': 'pdf'}, page_content='Importance  of  SAFERTOS  (Safety-Critical  \nReal-Time\n \nOS)\n \nSAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  \nreliability,\n \nand\n \ncertification\n \nmatter\n \n(automotive,\n \nmedical\n \ndevices,\n \nindustrial\n \nsafety).\n \n \nIt\n \nis\n \nderived\n \nfrom\n \nFreeRTOS,\n \nbut\n \nrewritten\n \nby\n \nsafety\n \nexperts\n \nand\n \ncertified\n \nto\n \nIEC\n \n61508\n \nSIL3,\n \nISO\n \n26262\n \nASIL\n \nD,\n \nIEC\n \n62304\n \nClass\n \nC,\n \nand\n \nmore.\n \n1)  SAFERTOS  for  Automotive  and  Supported  Processors  \nWhy  SAFERTOS  Is  Used  in  Automotive  Systems  \nAutomotive\n \n

In [45]:
embeddings=embedding_manager.generate_embeddings([doc.page_content for doc in chunks])
vector_store.add_documents(chunks,embeddings)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.71it/s]

Generated 6 embeddings
adding 6 documents to vector store
Successfully added 6 documents to vector store


### retreival testing 

In [46]:
### Retrieval function

In [48]:
def retrieve_relevant_chunks(query: str, vector_store: VectorStore, embedding_manager: EmbeddingManager, k: int = 3):
    """Retrieve relevant chunks for a given query"""
    # Generate embedding for the query
    query_embedding = embedding_manager.generate_embeddings([query])[0]
    
    # Search in vector store
    results = vector_store.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Format results
    retrieved_chunks = []
    for i in range(len(results["documents"][0])):
        chunk_info = {
            "content": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity_score": 1 - results["distances"][0][i]  # Convert distance to similarity
        }
        retrieved_chunks.append(chunk_info)
    
    return retrieved_chunks

# Test retrieval
test_query = "What is SAFERTOS used for in automotive systems?"
retrieved_docs = retrieve_relevant_chunks(test_query, vector_store, embedding_manager)
print(f"Retrieved {len(retrieved_docs)} chunks for query: '{test_query}'")
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Chunk {i+1} (Similarity: {doc['similarity_score']:.3f}) ---")
    print(f"Content: {doc['content'][:200]}...")
    print(f"Source: {doc['metadata'].get('source_file', 'unknown')}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.96it/s]

Generated 1 embeddings
Retrieved 3 chunks for query: 'What is SAFERTOS used for in automotive systems?'

--- Chunk 1 (Similarity: 0.360) ---
Content: Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
...
Source: Untitled document (4) (1).pdf

--- Chunk 2 (Similarity: 0.360) ---
Content: Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
...
Source: Untitled document (4) (1).pdf

--- Chunk 3 (Similarity: 0.258) ---
Content: ●  MPC57xx  
 
TI  Hercules  ARM  Cortex-R  
●  Common  in  safety-critical  automotive  subsystems.  
 
Infineon  AURIX  
●  Powertrain,  autonomous  drive  safety  modules.  
 
2)  SAFERTOS  in  Med...
Source: Untitled document (4) (1).pdf


### Augmentation

In [ ]:
print("Checking remaining cells...")

Augmented prompt created:
You are a helpful assistant that answers questions based on the provided context. Use only the information from the context to answer the question. If the context doesn't contain enough information to answer the question, say so.

CONTEXT:

--- Document 1 ---
Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
 
matter
 
(automotive,
 
medical
 
devi...


In [52]:
import ollama

class RAGPipeline:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager, model_name: str = "mistral"):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        self.model_name = model_name
    
    def query(self, question: str, k: int = 3):
        """Complete RAG pipeline: retrieve -> augment -> generate"""
        print(f"🔍 Processing query: '{question}'")
        
        # Step 1: Retrieve relevant chunks
        print("📚 Retrieving relevant documents...")
        retrieved_chunks = retrieve_relevant_chunks(question, self.vector_store, self.embedding_manager, k)
        
        # Step 2: Create augmented prompt
        print("✍️ Creating augmented prompt...")
        augmented_prompt = create_augmented_prompt(question, retrieved_chunks)
        
        # Step 3: Generate response using Ollama
        print(f"🤖 Generating response using {self.model_name}...")
        try:
            response = ollama.chat(
                model=self.model_name,
                messages=[{"role": "user", "content": augmented_prompt}]
            )
            answer = response['message']['content']
            
            # Step 4: Return results
            return {
                "question": question,
                "answer": answer,
                "sources": retrieved_chunks
            }
        except Exception as e:
            return {
                "question": question,
                "answer": f"Error generating response: {e}",
                "sources": retrieved_chunks
            }

# Initialize RAG pipeline
rag_pipeline = RAGPipeline(vector_store, embedding_manager, "mistral")
print("RAG pipeline initialized with Mistral!")

RAG pipeline initialized with Mistral!


In [ ]:
### Test RAG Pipeline

In [53]:
# Test with a sample question
question = "What are the key certifications of SAFERTOS and why are they important?"
result = rag_pipeline.query(question)

print("=" * 60)
print("QUESTION:", result["question"])
print("=" * 60)
print("ANSWER:")
print(result["answer"])
print("=" * 60)
print("SOURCES USED:")
for i, source in enumerate(result["sources"]):
    print(f"\nSource {i+1} (Similarity: {source['similarity_score']:.3f}):")
    print(f"File: {source['metadata'].get('source_file', 'unknown')}")
    print(f"Page: {source['metadata'].get('page', 'unknown')}")
    print(f"Content: {source['content'][:150]}...")

🔍 Processing query: 'What are the key certifications of SAFERTOS and why are they important?'
📚 Retrieving relevant documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.57it/s]

Generated 1 embeddings
✍️ Creating augmented prompt...
🤖 Generating response using mistral...


QUESTION: What are the key certifications of SAFERTOS and why are they important?
ANSWER:
 The key certification for SAFERTOS is IEC 62304 Class C. This certification is important because it is the highest safety classification for medical software and is required in life-sustaining equipment, surgical robots, implantable devices, and diagnostic machines in medical applications. It ensures extreme reliability, repeatability, and certified software safety which are critical requirements in such devices.
SOURCES USED:

Source 1 (Similarity: 0.168):
File: Untitled document (4) (1).pdf
Page: 2
Content: ●  MPC57xx  
 
TI  Hercules  ARM  Cortex-R  
●  Common  in  safety-critical  automotive  subsystems.  
 
Infineon  AURIX  
●  Powertrain,  autonomous ...

Source 2 (Similarity: 0.168):
File: Untitled document (4) (1).pdf
Page: 2
Content: ●  MPC57xx  
 
TI  Hercules  ARM  Cortex-R  
●  Common  in  safety-critical  automotive  subsystems.  
 
Infineon  AURIX  
●  Powertrain,  autonomous ...

S

In [54]:
def create_augmented_prompt(query: str, retrieved_chunks: list, context_limit: int = 2000):
    """Create an augmented prompt with retrieved context"""
    
    # Combine retrieved chunks with context limit
    context = ""
    for i, chunk in enumerate(retrieved_chunks):
        if len(context) + len(chunk["content"]) <= context_limit:
            context += f"\n--- Document {i+1} ---\n{chunk['content']}\n"
        else:
            # Add partial chunk if space allows
            remaining_space = context_limit - len(context)
            if remaining_space > 100:  # Only add if we have meaningful space
                context += f"\n--- Document {i+1} (truncated) ---\n{chunk['content'][:remaining_space-50]}...\n"
            break
    
    # Create the augmented prompt
    augmented_prompt = f"""You are a helpful assistant that answers questions based on the provided context. Use only the information from the context to answer the question. If the context doesn't contain enough information to answer the question, say so.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
    
    return augmented_prompt

# Test augmentation
test_prompt = create_augmented_prompt(test_query, retrieved_docs)
print("Augmented prompt created:")
print("=" * 50)
print(test_prompt[:500] + "..." if len(test_prompt) > 500 else test_prompt)

Augmented prompt created:
You are a helpful assistant that answers questions based on the provided context. Use only the information from the context to answer the question. If the context doesn't contain enough information to answer the question, say so.

CONTEXT:

--- Document 1 ---
Importance  of  SAFERTOS  (Safety-Critical  
Real-Time
 
OS)
 
SAFERTOS  is  a  pre-certified,  safety-critical  RTOS  designed  for  systems  where  determinism,  
reliability,
 
and
 
certification
 
matter
 
(automotive,
 
medical
 
devi...
